<!-- nav -->
[← 1 · Constraints as data](01_Constraints_As_Data.ipynb) · [Home](../../README.md) · [3 · Constraints over collections →](03_Constraints_Over_Collections.ipynb)

# 2 · Answers you can trust

**The problem.** Loggers drop out: a battery dies, a radio loses signal, and a shipment arrives with no temperature on
file. What should "is it in range?" answer then? A TypeScript function compares `undefined` with `2.0` and says
`false`: the shipment is quietly rejected, or, with the test written the other way round, quietly accepted. A Python
one raises `TypeError`. Neither is an answer you can act on.

An expression answers **unknown** when the data doesn't decide, and says so. This case study shows how unknowns
combine, how you decide what they mean, and why `1n` and `1.0` aren't equal.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import { cold_chain, show } from "./toolkit.ts";

const store = cold_chain();
const s1 = store.Shipment().id("S1").celsius(5.5).hours(30n).create();
const s2 = store.Shipment().id("S2").celsius(9.1).hours(80n).create();
const s3 = store.Shipment().id("S3").hours(20n).create(); // its logger never reported
const shipments = [s1, s2, s3];
const self = E.variable("this");
const celsius = self.celsius;
const in_range = E.literal(2.0).le(celsius).and_(celsius.le(8.0));
const table = (constraint: any) => Object.fromEntries(shipments.map((s: any) => [s.id, Evaluators.predicate(constraint, s)]));

## Step 1: absent is unknown

In mbse-schemas, a property is either present or absent; there's no `undefined` value to compare. Reading an absent
property gives **unknown**, which TypeScript writes `null`, and a comparison with unknown is unknown:

In [2]:
console.log(table(in_range));

{ S1: true, S2: false, S3: null }


S3's answer isn't false: it's that nobody knows. `has` asks whether a property is present, which always has an
answer:

In [3]:
const reported = self.has("celsius");
console.log(show(reported));
console.log(table(reported));

has(this, 'celsius')


{ S1: true, S2: true, S3: false }


## Step 2: how unknowns combine

`and_`, `or_`, `not_` and `implies` follow **Kleene's three-valued logic**: an unknown operand doesn't matter when the
other one decides. `false and unknown` is false, whatever the unknown turns out to be; `true or unknown` is true.

In [4]:
const late_or_out = self.hours.gt(72n).or_(in_range.not_());
const ok_or_silent = self.has("celsius").not_().or_(in_range);
for (const constraint of [late_or_out, ok_or_silent]) {
  console.log(show(constraint));
  console.log(table(constraint));
}

or(gt(get(this, 'hours'), 72), not(and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))))


{ S1: false, S2: true, S3: null }


or(not(has(this, 'celsius')), and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)))


{ S1: true, S2: false, S3: true }


For S3, the first constraint is unknown: it was on time, so everything rests on a temperature nobody has. The second
decides S3 at its first operand, and doesn't evaluate the second.

`implies` reads "if ... then": a constraint about only some shipments holds trivially for the others:

In [5]:
const long_trips_cold = self.hours.gt(72n).implies(self.celsius.le(8.0));
console.log(show(long_trips_cold));
console.log(table(long_trips_cold));

implies(gt(get(this, 'hours'), 72), le(get(this, 'celsius'), 8.0))


{ S1: true, S2: false, S3: true }


## Step 3: you decide what unknown means

An evaluator never guesses. The program asking decides, once, at the point where it acts. A dashboard that flags
shipments to inspect flags what isn't known to be fine; an audit that certifies shipments certifies only what is known
to be fine:

In [6]:
const inspect = shipments.filter((s: any) => Evaluators.predicate(in_range, s) !== true).map((s: any) => s.id);
const certify = shipments.filter((s: any) => Evaluators.predicate(in_range, s) === true).map((s: any) => s.id);
console.log(inspect, certify);

[ "S2", "S3" ] [ "S1" ]


Writing `if (Evaluators.predicate(...))` would treat unknown as false, which is the silent mistake this case study
started with. Compare with `=== true` or `=== false`.

## Step 4: no coercion

An integer and a float are different kinds of value, as they are in mbse-schemas: `1n` and `1.0` aren't equal, they're
incomparable, so comparing them is unknown. That catches a classic slip. `celsius` is a float, so a constraint that
writes the limits as integers answers unknown for every shipment, rather than an answer that depends on the language:

In [7]:
const sloppy = E.literal(2n).le(self.celsius).and_(self.celsius.le(8n));
console.log(table(sloppy));

{ S1: null, S2: null, S3: null }


JavaScript would have compared `2n <= 5.5` happily, and Python compares `2 <= 5.5` too, each by its own rules. A
constraint meant to give the same answer everywhere can't depend on either. Arithmetic is stricter still: adding an
integer and a float is an error, raised when the constraint is evaluated:

In [8]:
try {
  Evaluators.OfAny(self.hours.add(0.5).gt(72.0), { this: s1 });
} catch (error) {
  console.log(String(error));
}

TypeError: add expects numbers of one domain, got int and float


## Step 5: what raises

Unknown is for data that doesn't decide. A constraint that can't be evaluated at all raises: a variable nothing binds,
an operation outside the core vocabulary, an operand of the wrong kind. Those are mistakes in the constraint, not in the
data, and `validate` (case study 1) reports the first two before the constraint is evaluated.

In [9]:
for (const [expression, scope] of [[in_range, {}], [E.operation("div", 1n, 2n), {}], [self.id.add(1n), { this: s1 }]] as const) {
  try {
    Evaluators.OfAny(expression, scope);
  } catch (error) {
    console.log(String(error));
  }
}

KeyError: variable 'this' is not bound


NotImplementedError: 'div' is not a core operation


TypeError: add expects numbers of one domain, got str and int


## Why it works this way

- **Unknown is not false.** Treating it as false fails good records silently, and treating it as true passes bad ones.
  Three values keep the question open until a program, knowing what it's for, closes it.
- **Kleene, because it's the logic of not knowing yet.** `false and unknown` is false because no value of the unknown
  could make it true. Every answer an expression gives is one that more data couldn't overturn.
- **No coercion, because languages disagree.** TypeScript, Python, MATLAB, Excel and C each convert numbers by their
  own rules. A constraint that relied on any of them would mean different things in different places.

<!-- nav -->
[← 1 · Constraints as data](01_Constraints_As_Data.ipynb) · [Home](../../README.md) · [3 · Constraints over collections →](03_Constraints_Over_Collections.ipynb)